In [12]:
import pandas as pd
import kagglehub


# IEEE-CIS Fraud Detection

Join key is `TransactionID`. Transaction table has the label `isFraud` and payment features; identity table has device / id_* fields 


## Download

In [13]:
import shutil
from pathlib import Path

ieee_repo = Path("../../data/raw/ieee-fraud-detection")
ieee_cache = Path(kagglehub.competition_download("ieee-fraud-detection"))
print("Path to competition files:", ieee_cache)

skip_names = {"sample_submission.csv"}
ieee_repo.mkdir(parents=True, exist_ok=True)
for src in ieee_cache.rglob("*"):
    if src.is_file() and src.name not in skip_names:
        dest = ieee_repo / src.relative_to(ieee_cache)
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists():
            shutil.copy2(src, dest)
        print(dest.name, dest.stat().st_size)

leftover = ieee_repo / "sample_submission.csv"
if leftover.exists():
    leftover.unlink()


Path to competition files: /home/martinezyamamotoarthur/.cache/kagglehub/competitions/ieee-fraud-detection
train_transaction.csv 683351067
test_identity.csv 25797161
train_identity.csv 26529680
test_transaction.csv 613194934


In [14]:
IEEE_DIR = Path("../../data/raw/ieee-fraud-detection")
skip_names = {"sample_submission.csv"}
ieee_files = (
    sorted(p for p in IEEE_DIR.glob("*") if p.is_file() and p.name not in skip_names)
    if IEEE_DIR.exists()
    else []
)

if not ieee_files:
    print(
        "IEEE files not found. In download_data.ipynb:\n"
        "  1. Put kaggle.json in ~/.kaggle/\n"
        "  2. Accept https://www.kaggle.com/competitions/ieee-fraud-detection/rules\n"
        "  3. Run the competition_download cell\n"
        f"Expected dir: {IEEE_DIR.resolve()}"
    )
else:
    print(f"dir={IEEE_DIR.resolve()}")
    for p in ieee_files:
        print(f"  {p.name:28} {p.stat().st_size / 1e6:8.1f} MB")

    ieee_preview = {
        p.stem: pd.read_csv(p, nrows=5)
        for p in ieee_files
        if p.suffix == ".csv"
    }
    pd.set_option("display.max_columns", 40)
    for name, df in ieee_preview.items():
        print(f"\n=== {name}  cols={len(pd.read_csv(IEEE_DIR / f'{name}.csv', nrows=0).columns)} ===")
        display(df)

    tx_path = IEEE_DIR / "train_transaction.csv"
    if tx_path.exists():
        ieee_train_tx = pd.read_csv(
            tx_path,
            usecols=["TransactionID", "isFraud", "TransactionDT", "TransactionAmt", "ProductCD"],
        )
        id_path = IEEE_DIR / "train_identity.csv"
        n_identity = (
            pd.read_csv(id_path, usecols=["TransactionID"]).shape[0] if id_path.exists() else 0
        )
        print(
            f"train_transaction: n={len(ieee_train_tx):,}  "
            f"fraud_rate={ieee_train_tx['isFraud'].mean():.4%}  "
            f"amt_min={ieee_train_tx['TransactionAmt'].min()}  "
            f"amt_max={ieee_train_tx['TransactionAmt'].max()}  "
            f"DT min/max={ieee_train_tx['TransactionDT'].min()}/{ieee_train_tx['TransactionDT'].max()}"
        )
        print(f"train_identity rows={n_identity:,}  "
              f"identity coverage={n_identity / len(ieee_train_tx):.1%}")
        print(ieee_train_tx["ProductCD"].value_counts())


dir=/home/martinezyamamotoarthur/modeling_fraud_system/data/raw/ieee-fraud-detection
  test_identity.csv                25.8 MB
  test_transaction.csv            613.2 MB
  train_identity.csv               26.5 MB
  train_transaction.csv           683.4 MB

=== test_identity  cols=41 ===


,TransactionID,id-01,id-02,id-03,id-04,id-05,id-06,id-07,id-08,id-09,id-10,id-11,id-12,id-13,id-14,id-15,id-16,id-17,id-18,id-19,...,id-21,id-22,id-23,id-24,id-25,id-26,id-27,id-28,id-29,id-30,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663586,-45.0,280290.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,100.0,NotFound,27.0,NaN,New,NotFound,225.0,15.0,427.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
1,3663588,0.0,3579.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,100.0,Found,NaN,-300.0,Found,Found,166.0,NaN,542.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,Android 6.0.1,chrome 67.0 for android,24.0,1280x720,match_status:2,T,F,T,T,mobile,LGLS676 Build/MXB48T
2,3663597,-5.0,185210.0,NaN,NaN,1.0,0.0,NaN,NaN,NaN,NaN,100.0,NotFound,52.0,-360.0,New,NotFound,225.0,NaN,271.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,ie 11.0 for tablet,NaN,NaN,NaN,F,T,T,F,desktop,Trident/7.0
3,3663601,-45.0,252944.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,27.0,NaN,Found,Found,225.0,15.0,427.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,MYA-L13 Build/HUAWEIMYA-L13
4,3663602,-95.0,328680.0,NaN,NaN,7.0,-33.0,NaN,NaN,NaN,NaN,100.0,NotFound,27.0,NaN,New,NotFound,225.0,15.0,567.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 67.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,SM-G9650 Build/R16NW



=== test_transaction  cols=393 ===


,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,...,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,3663549,18403224,31.95,W,10409,111.0,150.0,visa,226.0,debit,170.0,87.0,1.0,NaN,gmail.com,NaN,6.0,6.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.00,W,4272,111.0,150.0,visa,226.0,debit,299.0,87.0,4.0,NaN,aol.com,NaN,3.0,2.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.00,W,4476,574.0,150.0,visa,226.0,debit,472.0,87.0,2635.0,NaN,hotmail.com,NaN,2.0,2.0,0.0,0.0,...,263.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.95,W,10989,360.0,150.0,visa,166.0,debit,205.0,87.0,17.0,NaN,gmail.com,NaN,5.0,2.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.95,W,18018,452.0,150.0,mastercard,117.0,debit,264.0,87.0,6.0,NaN,gmail.com,NaN,6.0,6.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



=== train_identity  cols=41 ===


,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,...,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,NaN,100.0,NotFound,49.0,-300.0,New,NotFound,166.0,NaN,621.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,52.0,NaN,Found,Found,121.0,NaN,410.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,NaN,100.0,NotFound,52.0,NaN,New,NotFound,225.0,NaN,176.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,100.0,NotFound,NaN,-300.0,Found,Found,166.0,15.0,529.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS



=== train_transaction  cols=394 ===


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,...,V320,V321,V322,V323,V324,V325,V326,V327,V328,V329,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,NaN,1.0,1.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,NaN,2.0,5.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


train_transaction: n=590,540  fraud_rate=3.4990%  amt_min=0.251  amt_max=31937.391  DT min/max=86400/15811131
train_identity rows=144,233  identity coverage=24.4%
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64


## Transaction ⋈ identity

`DataFrame.join(..., on="TransactionID")` looks up that column in the **left** frame against the **right** frame’s index, so identity must be indexed on `TransactionID`. Left join, then the share of train transactions that found an identity row.


In [ ]:
train_tx = pd.read_csv(IEEE_DIR / "train_transaction.csv", usecols=["TransactionID"])
id_tx = pd.read_csv(IEEE_DIR / "train_identity.csv", usecols=["TransactionID"]).assign(
    _has_identity=True
)

joined = train_tx.join(id_tx.set_index("TransactionID"), on="TransactionID", how="left")
matched = joined["_has_identity"].notna()

print(f"train_tx rows={len(train_tx):,}")
print(f"id_tx rows={len(id_tx):,}  unique TransactionID={id_tx['TransactionID'].nunique():,}")
print(f"matches={matched.sum():,}  match_rate={matched.mean():.2%}")
print(f"unmatched train_tx={ (~matched).sum():,}  miss_rate={(~matched).mean():.2%}")

id_in_tx = id_tx["TransactionID"].isin(train_tx["TransactionID"])
print(f"id_tx keys found in train_tx={id_in_tx.mean():.2%}  ({id_in_tx.sum():,}/{len(id_tx):,})")


# IEEE-CIS — what the columns mean

Vesta published **families**, not a pairwise dictionary. Safe approach:

1. Rename only fields whose **values** (or the official writeup) make the meaning obvious.
2. Keep `C*`, most `D*`, `V*`, and most `id_*` codes as family prefixes — do not invent “C1 = address count”.

Interpretable map + `rename_ieee_columns()` live in `notebooks_and_exploration/ieee_column_dictionary.py`. Filter the same table in the [IEEE column dictionary](/home/martinezyamamotoarthur/.cursor/projects/home-martinezyamamotoarthur-modeling-fraud-system/canvases/ieee-column-dictionary.canvas.tsx) canvas.

**Five information types**

| Kind | Columns | What you know |
| --- | --- | --- |
| Payment & card | `TransactionAmt`, `ProductCD`, `card1–card6` | Amount, channel (W retail / C CNP / HRS digital), card token, issuer, issue country, network, BIN/product, debit vs credit |
| Location & email | `addr1/2`, `dist1/2`, `P_/R_emaildomain` | Billing region/country, distance (W vs digital), purchaser vs recipient email |
| Entity history | `C1–C14`, `D1–D15`, `M1–M9`, `V1–V339` | Counts, day recency (`D9` = time of day), match flags, Vesta graph scores |
| Device / network | identity table (~24% of train) | OS, browser, screen, timezone, seen-before, IP country, sparse proxy block |
| Label / time / key | `isFraud`, `TransactionDT`, `TransactionID` | Target; seconds from a hidden start (not a real timestamp); join key |

**`ProductCD` is a channel, not a SKU.** On full train: W has billing address + `dist1` + M matches and almost never a recipient email; C is missing `addr` ~95%, has `R_emaildomain`, and ~11.7% fraud.

In [19]:
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path("..").resolve()
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from ieee_column_dictionary import (
    column_frame,
    rename_ieee_columns,
    rename_map,
)

# Full dictionary (one row per original column, including V1–V339)
ieee_cols = column_frame()
readable = ieee_cols.loc[
    ~ieee_cols["original"].str.match(r"^V\d+$"),
    ["original", "renamed", "table", "family", "information", "meaning", "confidence"],
].copy()


def _table_for_samples(path, originals, nrows=12_000):
    if not Path(path).exists():
        return pd.DataFrame()
    header = set(pd.read_csv(path, nrows=0).columns)
    usecols = []
    for col in originals:
        if col in header:
            usecols.append(col)
        elif col.startswith("id_") and col.replace("id_", "id-") in header:
            usecols.append(col.replace("id_", "id-"))
    if not usecols:
        return pd.DataFrame()
    raw = pd.read_csv(path, usecols=usecols, nrows=nrows)
    raw.columns = [c.replace("-", "_") for c in raw.columns]
    return raw


def _fmt(value):
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    text = str(value)
    return text if len(text) <= 40 else text[:37] + "..."


def _examples(frame, col):
    if col not in frame.columns:
        return ""
    shown = []
    for value in frame[col].dropna():
        text = _fmt(value)
        if text not in shown:
            shown.append(text)
        if len(shown) == 3:
            break
    return ", ".join(shown)


tx_samples = _table_for_samples(
    IEEE_DIR / "train_transaction.csv",
    readable.loc[readable["table"] == "transaction", "original"],
)
id_samples = _table_for_samples(
    IEEE_DIR / "train_identity.csv",
    readable.loc[readable["table"] == "identity", "original"],
)
readable["sample"] = [
    _examples(tx_samples if table == "transaction" else id_samples, original)
    for original, table in zip(readable["original"], readable["table"])
]
readable = readable[
    ["original", "renamed", "sample", "table", "family", "information", "meaning", "confidence"]
]

print(readable["confidence"].value_counts().to_string())
print("\nRecommended rename (high + medium only):")
print(pd.Series(rename_map("transaction", min_confidence="medium")).head(25).to_string())

# Apply to in-memory frames if you already loaded them:
# ieee_train_tx = rename_ieee_columns(ieee_train_tx, "transaction", min_confidence="medium")
# ieee_train_id = rename_ieee_columns(ieee_train_id, "identity", min_confidence="medium")

confidence
low       48
high      29
medium    19

Recommended rename (high + medium only):
TransactionID                       transaction_id
isFraud                                   is_fraud
TransactionDT               seconds_from_reference
TransactionAmt                          amount_usd
ProductCD                          product_channel
card1                                      card_id
card2                                  card_issuer
card3                           card_issue_country
card4                                 card_network
card5                          card_bin_or_product
card6                            card_funding_type
addr1                               billing_region
addr2                              billing_country
dist1             distance_billing_to_counterparty
dist2                       distance_alt_addresses
P_emaildomain               purchaser_email_domain
R_emaildomain               recipient_email_domain
D1                      days_since_card_f

In [18]:
from IPython.display import HTML, display

df = readable.query("confidence != 'low'")

display(HTML(f"""
<div style="max-height:500px; overflow:auto;">
    {df.to_html(index=False)}
</div>
"""))

original,renamed,sample,table,family,information,meaning,confidence
TransactionID,transaction_id,"2987000, 2987001, 2987002",transaction,keys,Join key to the identity table.,Unique transaction identifier.,high
isFraud,is_fraud,"0, 1",transaction,label,Competition target.,1 if Vesta labeled the payment as fraud.,high
TransactionDT,seconds_from_reference,"86400, 86401, 86469",transaction,time,"When the payment happened, relative to a hidden start date.",Seconds since a fixed reference (not a Unix timestamp). Train starts at 86400 (1 day).,high
TransactionAmt,amount_usd,"68.5, 29, 59",transaction,payment,How much was charged.,Transaction amount in USD.,high
ProductCD,product_channel,"W, H, C",transaction,payment,What kind of purchase / acceptance channel this is.,"W ≈ card-present / retail (has billing addr, dist1, M matches; no recipient email). C ≈ card-not-present / cross-border e-comm (addr missing ~95%, has recipient email, ~12% fraud). H/R/S ≈ digital / remote products (no dist1; H and R have identity rows more often).",high
card1,card_id,"13926, 2755, 4663",transaction,payment card,Which payment card was used (anonymized).,Highest-cardinality card token. Closest thing to a card account id.,high
card2,card_issuer,"404, 490, 567",transaction,payment card,Issuing bank / BIN grouping.,Medium-cardinality numeric code (~500 values). Likely issuer or BIN fragment.,medium
card3,card_issue_country,"150, 117, 185",transaction,payment card,Country that issued the card.,150 dominates (US-like). 185 is the main foreign code and lines up with ProductCD=C.,high
card4,card_network,"discover, mastercard, visa",transaction,payment card,Card scheme.,visa / mastercard / american express / discover.,high
card5,card_bin_or_product,"142, 102, 166",transaction,payment card,"Card product / BIN category (classic vs platinum, etc.).",Low-cardinality numeric (~85). Not the same as card2.,medium


In [21]:
train_tx = pd.read_csv(IEEE_DIR / "train_transaction.csv")
id_tx = pd.read_csv(IEEE_DIR / "train_identity.csv")

In [25]:
train_tx[["isFraud"]].mean()

isFraud    0.03499
dtype: float64

In [28]:
joined = train_tx.join(id_tx.set_index("TransactionID"), on="TransactionID", how="inner")
joined.head(1100)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,...,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
4,2987004,0,86506,50.000,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
8,2987008,0,86535,15.000,H,2803,100.0,150.0,visa,226.0,debit,337.0,87.0,NaN,NaN,anonymous.com,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,iOS 11.1.2,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
10,2987010,0,86549,75.887,C,16496,352.0,117.0,mastercard,134.0,credit,NaN,NaN,NaN,NaN,gmail.com,gmail.com,1.0,4.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
11,2987011,0,86555,16.495,C,4461,375.0,185.0,mastercard,224.0,debit,NaN,NaN,NaN,30.0,hotmail.com,hotmail.com,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
16,2987016,0,86620,30.000,H,1790,555.0,150.0,visa,226.0,debit,170.0,87.0,NaN,NaN,aol.com,NaN,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,Mac OS X 10_11_6,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5636,2992636,0,179211,52.505,C,9917,142.0,185.0,visa,138.0,debit,NaN,NaN,NaN,631.0,gmail.com,gmail.com,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
5642,2992642,0,179319,52.505,C,3154,408.0,185.0,mastercard,224.0,debit,NaN,NaN,NaN,631.0,gmail.com,gmail.com,2.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Found,Found,NaN,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
5646,2992646,1,179385,200.000,R,16075,514.0,150.0,mastercard,102.0,credit,441.0,87.0,NaN,NaN,gmail.com,gmail.com,3.0,2.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,chrome 62.0 for android,32.0,1280x720,match_status:2,T,F,T,F,mobile,LGMP260 Build/NRD90U
5648,2992648,0,179413,19.153,C,15885,545.0,185.0,visa,138.0,debit,NaN,NaN,NaN,NaN,gmail.com,gmail.com,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,NaN,chrome 62.0 for android,NaN,NaN,NaN,F,F,T,T,mobile,LG-K580 Build/MRA58K


In [29]:
joined[["isFraud"]].mean()

isFraud    0.07847
dtype: float64

## Sweetviz

HTML reports for all four IEEE tables go under `reports/sweetviz/`. Transaction tables are randomly sampled (`40_000` rows) so the 390+ column profiles finish; identity tables are profiled in full. Train `isFraud` is the Sweetviz target. Test identity columns are normalized (`id-01` → `id_01`) so they match train. Pairwise associations are off on the wide transaction tables.


In [ ]:
import sweetviz as sv

SV_DIR = Path("../../reports/sweetviz")
SV_DIR.mkdir(parents=True, exist_ok=True)

TX_SAMPLE = 40_000
SEED = 42
IEEE_TABLES = (
    "train_transaction",
    "test_transaction",
    "train_identity",
    "test_identity",
)


def normalize_ieee_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.replace("-", "_") for c in df.columns]
    return df


def maybe_sample(df: pd.DataFrame, n: int = TX_SAMPLE) -> pd.DataFrame:
    if len(df) > n:
        return df.sample(n=n, random_state=SEED)
    return df


ieee_dfs = {}
for name in IEEE_TABLES:
    path = IEEE_DIR / f"{name}.csv"
    if not path.exists():
        print(f"skip {name}: {path} missing")
        continue

    print(f"Loading {name} from {path} ...")
    df = normalize_ieee_columns(pd.read_csv(path))
    if "transaction" in name:
        df = maybe_sample(df)
    ieee_dfs[name] = df

    pairwise = "off" if df.shape[1] > 80 else "auto"
    target = "isFraud" if "isFraud" in df.columns else None
    print(f"  sweetviz {name}: shape={df.shape} target={target} pairwise={pairwise}")
    report = sv.analyze(df, target_feat=target, pairwise_analysis=pairwise)
    html_path = SV_DIR / f"{name}.html"
    report.show_html(str(html_path), open_browser=False)
    print(f"  wrote {html_path.resolve()}")
    report.show_notebook(w="100%", h="720")

if {"train_transaction", "test_transaction"} <= ieee_dfs.keys():
    train_tx = ieee_dfs["train_transaction"].drop(columns=["isFraud"], errors="ignore")
    test_tx = ieee_dfs["test_transaction"]
    cols = train_tx.columns.intersection(test_tx.columns)
    print(f"Compare transaction train vs test on {len(cols)} columns")
    cmp_tx = sv.compare(
        [train_tx[cols], "train"],
        [test_tx[cols], "test"],
        pairwise_analysis="off",
    )
    cmp_tx_path = SV_DIR / "compare_transaction_train_test.html"
    cmp_tx.show_html(str(cmp_tx_path), open_browser=False)
    print(f"  wrote {cmp_tx_path.resolve()}")
    cmp_tx.show_notebook(w="100%", h="720")

if {"train_identity", "test_identity"} <= ieee_dfs.keys():
    train_id = ieee_dfs["train_identity"]
    test_id = ieee_dfs["test_identity"]
    cols = train_id.columns.intersection(test_id.columns)
    print(f"Compare identity train vs test on {len(cols)} columns")
    cmp_id = sv.compare(
        [train_id[cols], "train"],
        [test_id[cols], "test"],
        pairwise_analysis="auto",
    )
    cmp_id_path = SV_DIR / "compare_identity_train_test.html"
    cmp_id.show_html(str(cmp_id_path), open_browser=False)
    print(f"  wrote {cmp_id_path.resolve()}")
    cmp_id.show_notebook(w="100%", h="720")
